# Hard-Negative Mining and Debiased Contrastive Training (ANCE)

This notebook walks the companion module `negative_sampling_hard_negatives.py` section by section. The
module owns every number; here we narrate the four movements and watch the claims run.

InfoNCE taught us that the contrastive gradient on a negative *is* its softmax weight, so the hardest
negative dominates and random negatives are near-orthogonal noise. This topic spends that fact and pays
its hidden cost:

1. **Why hard negatives** — the few same-sector hard negatives carry a disproportionate share of the
   gradient, and a hard-mined batch is a larger learning signal than a random one.
2. **The false-negative problem** — mining *near* the anchor is mining where unlabeled true positives
   live, so the mined "negatives" are contaminated at a rate $\tau^+$.
3. **The debiased contrastive estimator** (Chuang et al., 2020) — the rigorous spine: recover the
   true-negative expectation from unlabeled samples.
4. **ANCE** (Xiong et al., 2021) — the systems object: global hard negatives mined from an index that
   goes *stale* as the encoder drifts.

In [ ]:
import numpy as np
import negative_sampling_hard_negatives as nsh

s = nsh._setup()
pool = s["pool"]
print(f"labeled mining pool: {pool['emb'].shape[0]} queries, "
      f"{len(np.unique(pool['company']))} companies, {len(np.unique(pool['sector']))} sectors")
print(f"class prior tau+ (random false-negative rate) = {s['tau_plus']:.4f}")

## Movement 1 — why hard negatives: the gradient weights by similarity

The InfoNCE gradient on a negative is its softmax weight $p_i = e^{s_i/\tau} / \sum_j e^{s_j/\tau}$. Because
the weight is exponential in the similarity, the *few* same-sector hard negatives in a batch carry a
share of the gradient far above their count fraction — and a hard-mined batch produces a larger loss
(more signal per step) than a random one. This generalizes the prerequisite's
`finance_hard_negative_share` to a mined batch.

In [ ]:
cos, same = nsh.negative_set(pool, nsh.M1_ANCHOR)
share = nsh.hard_gradient_share(cos, same, nsh.GRAD_TAU)
frac = float(np.mean(same))
print(f"{int(same.sum())} same-sector hard negatives of {len(cos)} ({frac:.1%} of the batch)")
print(f"they carry {share:.1%} of the gradient at tau={nsh.GRAD_TAU}")
print("hard share rises as tau falls:")
for r in nsh.hard_share_curve(cos, same, (1.0, 0.2, 0.05, 0.01)):
    print(f"   tau={r['tau']:>4}: {r['hard_share']:.3f}")
print("batch loss:", nsh.batch_loss_comparison(pool, nsh.M1_ANCHOR, nsh.M1_N_NEG, nsh.GRAD_TAU, nsh.M1_SEED))

## Movement 2 — the false-negative problem

Movement 1's recipe says *sample near the anchor*. But near the anchor is exactly where unlabeled true
positives live. A label-unaware miner that takes the nearest candidates therefore surfaces accidental
positives — false negatives — at a rate that is highest at the smallest mining depth, while uniform
random sampling only ever hits the global class prior. The substrate is load-bearing: the finance pool
has four queries per company, so same-company duplicates exist and the phenomenon is measurable.

In [ ]:
print(f"{'k':>3} {'mined tau+':>11} {'random tau+':>12} {'prior':>7}")
for r in nsh.tau_plus_curve(pool, nsh.K_MINE_GRID, nsh.M2_SEED):
    print(f"{r['k']:>3} {r['mined']:>11.3f} {r['random']:>12.3f} {r['prior']:>7.3f}")

## Movement 3 — the debiased contrastive estimator (the rigorous spine)

The unlabeled sampling law decomposes $p = \tau^+ p^+ + \tau^- p^-$, so the true-negative expectation is
recoverable from unlabeled samples plus the positive distribution:

$$\mathbb{E}_{p^-}[g] = \frac{\mathbb{E}_p[g] - \tau^+\, \mathbb{E}_{p^+}[g]}{\tau^-}, \qquad g(x) = e^{s(\text{anchor}, x)/\tau}.$$

At the full pool, with the empirical class prior, this is an **exact identity**. Under finite sampling
the debiased estimator beats the biased in-batch mean at every $N$ and converges toward the oracle,
while the biased mean plateaus at the contamination bias. Robinson et al.'s $\beta$-reweighting
$q_\beta \propto e^{\beta s}$ concentrates the estimator on harder negatives — and at $\beta = 1/\tau$ it *is*
the InfoNCE weighting, the bridge back to Movement 1.

In [ ]:
g, is_pos = nsh.estimator_g(pool, nsh.M3_ANCHOR, nsh.M3_TAU)
oracle = nsh.oracle_true_negative_mean(g, is_pos)
biased = nsh.biased_negative_mean(g)
pos_mean = nsh.positive_mean(g, is_pos)
tau_plus = float(np.mean(is_pos))
debiased = nsh.debiased_negative_mean(g, pos_mean, tau_plus, nsh.M3_TAU)
print(f"E_p+[g] = {pos_mean:.2f}   E_p[g] (biased) = {biased:.2f}   E_p-[g] (oracle) = {oracle:.2f}")
print(f"debiased estimator = {debiased:.4f}  ==  oracle {oracle:.4f}  (exact identity at the full pool)")
print()
print("convergence under sampling (biased MAE flat, debiased MAE -> 0):")
for r in nsh.debiased_convergence_curve(pool, nsh.M3_ANCHOR, nsh.M3_TAU, s['tau_plus'],
                                        (8, 32, 128, 512), nsh.M3_TRIALS, nsh.M3_SEED):
    print(f"   N={r['N']:>4}: biased={r['biased_mae']:.3f}  debiased={r['debiased_mae']:.3f}")

In [ ]:
# beta-reweighting: beta = 1/tau recovers the imported InfoNCE weighting exactly.
s_neg = np.log(g) * nsh.M3_TAU
print("beta = 1/tau matches negative_weights:",
      np.allclose(nsh.beta_reweight_weights(s_neg, 1.0 / nsh.M3_TAU),
                  nsh.negative_weights(s_neg, nsh.M3_TAU), atol=1e-12))
print("reweighted negative mean by beta (0 = uniform, 1/tau = InfoNCE):",
      [round(nsh.beta_reweighted_negative_mean(g, s_neg, b), 2) for b in (0.0, 1.0, 1.0 / nsh.M3_TAU)])

## Movement 4 — ANCE: the asynchronous index and staleness (the systems-math)

To mine *global* hard negatives, ANCE retrieves them from an approximate-nearest-neighbor index — which
goes **stale** as the encoder drifts during training. We model the drift as a non-isometric
interpolation of the embedding space toward a seeded target, freeze the index at the last refresh, and
measure staleness as the top-$k$ overlap between the frozen index's mined set and the fresh encoder's.
Staleness decays with steps-since-refresh and drops the now-relevant document; the refresh interval $R$
trades staleness against re-encode cost $1/R$.

In [ ]:
q0, d0, gold, T = s['ance_queries'], s['ance_docs'], s['ance_gold'], s['drift_T']
print(f"ANCE corpus: {d0.shape[0]} documents, {q0.shape[0]} queries; "
      f"fresh gold recall@1 = {nsh._gold_recall_at_1(d0, q0, gold):.3f}")
print()
print(f"{'t':>3} {'overlap':>8} {'stale r@1':>10} {'fresh r@1':>10}")
for r in nsh.staleness_curve(q0, d0, gold, T, nsh.T_GRID, nsh.K_STALE, nsh.DRIFT_TAU):
    print(f"{r['t']:>3} {r['overlap']:>8.3f} {r['stale_gold_r1']:>10.3f} {r['fresh_gold_r1']:>10.3f}")

In [ ]:
print("refresh interval R trades staleness against re-encode cost 1/R:")
print(f"{'R':>3} {'avg staleness':>14} {'cost 1/R':>9}")
for r in nsh.refresh_interval_tradeoff(q0, d0, T, nsh.R_GRID, nsh.HORIZON, nsh.K_STALE, nsh.DRIFT_TAU):
    print(f"{r['R']:>3} {r['avg_staleness']:>14.3f} {r['reencode_cost']:>9.3f}")

## The verification harness

Every pedagogical claim above is an `assert` in the module. Running the whole harness is the regression
test — each `[ok]` line is a claim that held on the actual finance geometry.

In [ ]:
nsh._run_all()